# Lab 11: APIs and JSON

**DSA 405 · Week 11**

| | |
|---|---|
| **In class** | Friday, Nov 6 |
| **A11 due** | Thursday, Nov 12, 11:59 PM |
| **APIs used** | USGS earthquakes and GitHub (both free, no key needed) |
| **Also** | `permits_raleigh.json` |
| **This week** | **P3 due Thu Nov 5** · Bench Check 2 window · lightning talk sign-up opens · P4 introduced |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

An **API** (application programming interface) is a service that returns data directly,
in a documented format, when your code requests it. The usual format is **JSON**, a
plain-text format that stores data as name-value pairs and lists. If an API exists for
the data you want, use it instead of scraping. This table shows why:

| | Scraping | API |
|---|---|---|
| Structure | inferred from HTML | documented |
| Stability | breaks on redesign | versioned |
| Permission | interpreted from terms | explicit |
| Rate limits | guessed | published |
| Errors | HTTP 200 with wrong content | proper status codes |

Today we learn the four techniques that APIs require: **parameters** (settings you send
with a request), **pagination** (results arrive split across several pages), **rate
limits** (a cap on how many requests you may send in a period of time), and
**flattening nested JSON** (turning data stored as dictionaries inside dictionaries
into a flat table). Flattening is where most of the work is, so most of the lab is
about it.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, time, re, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users

HEADERS = {"User-Agent": "DSA405-student-coursework (NC State; teaching exercise)"}


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

---
# Part 1: Explore (in class)

## Task 1.1: A first API call

The USGS earthquake API is free, needs no key, and is well documented. It also returns
a clear error message when a request is written incorrectly.

An API request is a URL plus **parameters**: name-value pairs that say exactly what data
you want. If you pass the parameters to `requests` as a dict, `requests` builds the
query string (the part of the URL after the `?`) for you. This is safer than joining
the strings together by hand.

In [ ]:
USGS = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "geojson",
    "starttime": "2026-01-01",
    "endtime": "2026-01-08",
    "minmagnitude": 5.0,
}

r = requests.get(USGS, params=params, headers=HEADERS, timeout=30)
print("URL requested:", r.url)
print("status       :", r.status_code)

r.raise_for_status()
payload = r.json()

print("top-level keys:", list(payload.keys()))
print("events returned:", len(payload["features"]))
print("API's own count:", payload["metadata"]["count"])

The last two lines show two counts: the number of events you received, and the count
the API reports for its own response. Compare them. Whenever an API reports its own
record count, check it against the number of rows you actually received. The check takes
one line of code and can catch a download that stopped before all the records arrived.

## Task 1.2: Read the errors on purpose

API error messages describe what went wrong. In the two scraping weeks, a failed request
returned a page with no useful content and no message, so this is a real improvement. Break a
request on purpose and read the message that comes back.

In [ ]:
bad = requests.get(USGS, params={"format": "geojson", "starttime": "not-a-date"},
                   headers=HEADERS, timeout=30)
print("status:", bad.status_code)
print(bad.text[:400])

When a scraper requests a page whose layout has changed, it gets back HTTP 200
("success") and no useful content. This API request returned an error status code and a
message describing exactly what was wrong. That difference is the main reason to use an
API whenever one exists.

## Task 1.3: Flatten the nesting

**Nested** data is data stored as dictionaries inside other dictionaries. **GeoJSON**, the
format the USGS API returns, is JSON with a fixed layout for geographic data, and it is
nested three levels deep. Each event looks roughly like:

```
{"properties": {"mag":..., "place":..., "time":...},
 "geometry":   {"coordinates": [lon, lat, depth]},
 "id": "..."}
```

`pd.json_normalize` flattens nested dicts into columns with dotted names, one column per
field. It does **not** unpack lists, so you must handle `coordinates` yourself.

In [ ]:
quakes = pd.json_normalize(payload["features"])
print(quakes.shape)
print([c for c in quakes.columns][:14])
quakes[["properties.mag", "properties.place", "geometry.coordinates"]].head()

Fix the two things `json_normalize` did not handle:

1. Split `geometry.coordinates` into `lon`, `lat`, `depth`
2. `properties.time` is a Unix timestamp in **milliseconds**: the number of milliseconds
   since January 1, 1970. Convert it to a date.

In [ ]:
quakes[["lon", "lat", "depth"]] = pd.DataFrame(
    quakes["geometry.coordinates"].tolist(), index=quakes.index)

quakes["when"] = pd.to_datetime(quakes["properties.time"], unit="ms")

print(quakes[["properties.mag", "properties.place", "lat", "lon", "depth", "when"]].head())
print()
print("time range:", quakes.when.min(), "to", quakes.when.max())

Verify the conversion, because the unit can be wrong in two directions. If the API had sent
seconds and you had written `unit="ms"`, every date would come out in January 1970, and
Python would raise no error. The other direction is loud: writing `unit="s"` for these
millisecond values raises an error, because the dates land in the year 57990, which pandas
cannot store. Check that the date range in the result matches the window you requested.

In [ ]:
assert quakes.when.min() >= pd.Timestamp(params["starttime"]), "dates before the requested window"
assert quakes.when.max() <= pd.Timestamp(params["endtime"]) + pd.Timedelta(days=1), "dates after"
assert len(quakes) == payload["metadata"]["count"], "row count disagrees with the API"
print("all checks passed")

---
## Checkpoint: submit before leaving

1. What record count did the API report for its own response, and did it match the
   number of rows you received?
2. Paste the error message from the deliberately broken request.
3. The timestamps are in milliseconds. If an API had sent them in seconds and you had still
   written `unit="ms"`, what year would every date land in? Run
   `pd.to_datetime(quakes["properties.time"] // 1000, unit="ms")` to check.
4. One sentence: does your P3 source have an API you should use instead of scraping?

*Your answers.*

---
# Part 2: A11 (APIs & JSON)

Three tasks, and they make up the take-home half of the week.

## Task 2.1: Pagination and rate limits

The GitHub REST API needs no key for public data. Without signing in, the core API
allows **60 requests per hour**, and the search endpoint used here has its own, much
lower limit. (An **endpoint** is one specific URL that an API answers requests on.) The
response headers state the exact limit. Trust the headers over any documentation or
blog post, because the headers describe the limit that applies to you right now.

The task: collect the **first 100 public repositories** matching a search query you are
interested in. This takes more than one request, because the API caps `per_page` at 100
and splits the results across pages.

Requirements:

- Read the rate limit **from the response headers**, not from documentation
- Stop before you use up the limit
- Sleep between requests
- Detect the last page instead of guessing where it is

In [ ]:
GH = "https://api.github.com/search/repositories"

r = requests.get(GH, params={"q": "web scraping language:python", "per_page": 5},
                 headers=HEADERS, timeout=30)
r.raise_for_status()

# the API reports how many requests you have left in its headers, so read them first
for k in ["x-ratelimit-limit", "x-ratelimit-remaining", "x-ratelimit-reset"]:
    print(f"{k:26} {r.headers.get(k)}")

print()
print("total matches:", r.json()["total_count"])

In [ ]:
def gh_search(query, want=100, per_page=50, max_requests=4):
    """Collect up to `want` repositories without going over the rate limit."""
    rows, page = [], 1
    while len(rows) < want and page <= max_requests:
        resp = requests.get(GH, params={"q": query, "per_page": per_page, "page": page},
                            headers=HEADERS, timeout=30)
        if resp.status_code == 403:
            print("rate limited. Stopping.")
            break
        resp.raise_for_status()
        items = resp.json()["items"]
        if not items:
            break                      # no more pages
        rows.extend(items)
        remaining = resp.headers.get("x-ratelimit-remaining")
        print(f"page {page}: {len(items)} items, {remaining} requests left")
        page += 1
        time.sleep(1)
    return pd.json_normalize(rows[:want])


repos = gh_search("web scraping language:python")
print(repos.shape)

Report your verification: how many records did you request, how many arrived, and what
evidence confirmed that you reached the last page rather than stopping partway through
the results?

*Your answer.*

## Task 2.2: The nested file with inconsistent keys

This task uses a local file, so no rate limit applies.

`permits_raleigh.json` is a realistic export from a city government system. It contains
three problems. None of them can be fixed by code alone; for each one you first have to
decide what the data means.

In [ ]:
payload = load("permits_raleigh.json", "json")

print("metadata:", payload["metadata"])
print("records :", len(payload["results"]))
print()
import json as _json
print(_json.dumps(payload["results"][0], indent=1)[:500])

### 2.2a: The same concept under three different key names

The monetary value of a permit is stored under **three different keys**, depending on
which source system the record came from.

Find all three key names. Then count how many records you would lose if your code read
only one of them.

In [ ]:
from collections import Counter

keys = Counter()
for rec in payload["results"]:
    for k in rec["permit"]:
        keys[k] += 1
print(keys)

Report how many records have a value field under *some* name, how many have
`permit_value` specifically, and what percentage of records you would lose, with no
error message, by reading only `permit_value`.

*Your answer.*

### 2.2b: Five encodings, four meanings

Collect every distinct way this field writes "no value". (Here, an **encoding** means
one specific way of writing it, such as a special number or a special word.) There are
five encodings, and they have **four different meanings**.

Work out what each one means. You can discover the meaning from the data: check which
permit types and which months each encoding appears in. This task asks only for interpretation; the
cleanup comes after.

In [ ]:
VALUE_KEYS = ("permit_value", "value", "estimated_cost")

def get_value(rec):
    for k in VALUE_KEYS:
        if k in rec["permit"]:
            return rec["permit"][k]
    return "NO_KEY_AT_ALL"


vals = [get_value(r) for r in payload["results"]]
odd = [v for v in vals if not isinstance(v, int) or v <= 0]
print("distinct non-positive / non-numeric values:", sorted({repr(v) for v in odd}))

In [ ]:
# for each odd encoding, which permit types and which months does it appear in?
rows = []
for rec in payload["results"]:
    v = get_value(rec)
    rows.append({"type": rec["permit"]["type"],
                 "month": rec["permit"]["issued"][5:7],
                 "value": v,
                 "encoding": repr(v) if (not isinstance(v, int) or v <= 0) else "number"})
pv = pd.DataFrame(rows)

for enc in pv[pv.encoding != "number"].encoding.unique():
    sub = pv[pv.encoding == enc]
    print(f"\n{enc}  ({len(sub)} records)")
    print("  types :", sub.type.value_counts().head(3).to_dict())
    print("  months:", sorted(sub.month.unique()))

State what each of the five encodings means, with evidence. Two are concentrated in
specific permit types. One is concentrated in a specific date range, which suggests that
something happened in the source system during that period.

Then state which of the five should become `NaN` and which should not, with a
justification for each. Getting this wrong changes the total declared permit value in
the county.

*Your answer.*

### 2.2c: Missing keys and fields with two different formats

Two more problems. Code that does not expect them stops with an error:

1. Some records have **no `contractor` key at all**, so `rec["contractor"]["name"]`
   raises an error.
2. `location.coords` is a `[lat, lon]` **list** in most records and a `{"lat":…, "lon":…}`
   **dict** in some.

Handle both. Build a flat DataFrame with `permit_id`, `type`, `issued`, `value`, `city`,
`contractor_name`, `lat`, `lon`. The code must not crash, and it must not produce a
wrong value without an error message.

In [ ]:
# build the flat table

Verification:

```python
assert len(flat) == 600
assert flat.permit_id.is_unique
assert flat.lat.between(35.0, 36.5).all()
assert flat.lon.between(-79.5, -78.0).all()
```

Report how many records lacked a contractor and how many stored their coordinates as a
dict.

In [ ]:
# your assertions

## Task 2.3: APIs versus scraping, applied to your project

**150 words**, about your own project rather than APIs in general.

1. Does any of your data have an API you are not using? If yes, why not?
2. For the data you are scraping: what would an API provide that scraping cannot?
3. Name one thing you lose by using an API instead of scraping. There is at least one.

*Your answer.*

---
## AI use note

Tell me which AI tools you used here and what you used them for, in a sentence or two.
If you didn't use any, write "none."

*Your answer.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. Download the `.ipynb`, rename to `DSA405_002_FA26_A11_[yourUnityID].ipynb`
3. Upload to the **A11** space. The Checkpoint goes to **Week 11 In-Class Activity** before
   leaving class.

A11 is due **Thursday, Nov 12, 11:59 PM**.

**Also this week:** **P3 due Thursday, Nov 5** · Bench Check 2 window (Weeks 10–12) ·
lightning talk sign-up opens.